# 01 - Index / ETF registry audit

**What:** build and inspect the catalogue of products & benchmarks that could force buying/selling into ASX names.

**Why:** every downstream event needs a product with known metadata (provider, benchmark, theme, AUM, rebalance cadence, holdings access). This notebook audits coverage and confidence so we know where the gaps (and the manual drop-ins) are.

**How:** seeds (watchlists + providers) + discovery + manual CSVs -> one registry, scored by completeness.

In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import pandas as pd
from index_flow.config import load_config
from index_flow.registry import build_registry, split_and_save
cfg = load_config(); cfg.ensure_dirs()
reg = build_registry(cfg)
split_and_save(cfg, reg)
print(reg.shape); reg.head()

## Coverage & confidence
Where confidence is low we are missing holdings_url / AUM / # holdings / history. Those rows are candidates for a manual CSV in `data/manual/etf_registry/`.

In [ ]:
print('By record type:'); print(reg['record_type'].value_counts())
print('\nBy theme (ASX-exposed):')
print(reg[reg['asx_exposure_flag'].fillna(False).astype(bool)]['theme'].value_counts())
print('\nConfidence distribution:'); print(reg['confidence_score'].describe())
reg.sort_values('confidence_score').head(15)[['product_ticker','theme','source','confidence_score']]